# 02 — Обучение модели 1: ruBERT

**Модель:** `blanchefort/rubert-base-cased-sentiment`

ruBERT — BERT-модель, предобученная на русских текстах и дополнительно дообученная на задаче анализа тональности. Использует кириллический токенизатор.

**Гиперпараметры:**
- batch_size: 32 (T4 GPU: ~12 GB VRAM)
- learning_rate: 2e-5
- epochs: 5 (с early stopping)

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(f'{WORKING_DIR}/models/rubert', exist_ok=True)

print(f'Working dir: {WORKING_DIR}')

In [ ]:
import torch
import numpy as np
from datasets import load_from_disk, DatasetDict
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Загрузка обработанных данных

In [ ]:
# Загружаем датасет, сохранённый в ноутбуке 01
DATA_PATH = f'{WORKING_DIR}/processed_data'

if os.path.exists(DATA_PATH):
    dataset = load_from_disk(DATA_PATH)
    print(f'Загружен обработанный датасет из: {DATA_PATH}')
else:
    # Если ноутбук 01 не запускался — загружаем напрямую
    print('Обработанный датасет не найден, загружаю из источника...')
    from src.data_loader import load_data
    from src.preprocessor import preprocess_batch
    dataset, _ = load_data()
    dataset = dataset.map(
        lambda batch: preprocess_batch(batch, clean=True),
        batched=True, desc='Preprocessing'
    )

for split in dataset:
    print(f'  {split}: {len(dataset[split]):,} примеров')

## Обучение ruBERT

In [ ]:
from src.trainer import train_model

MODEL_NAME = 'blanchefort/rubert-base-cased-sentiment'
OUTPUT_DIR = f'{WORKING_DIR}/models/rubert'

model, tokenizer, results = train_model(
    model_name=MODEL_NAME,
    dataset=dataset,
    output_dir=OUTPUT_DIR,
    num_labels=3,
    num_epochs=5,
    batch_size=32,
    learning_rate=2e-5,
    max_length=128,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    seed=42,
    early_stopping_patience=2,
)

## Результаты

In [ ]:
from src.evaluation import evaluate_predictions, plot_confusion_matrix
import numpy as np

preds  = np.array(results['predictions'])
labels = np.array(results['true_labels'])
probs  = np.array(results['probabilities'])

metrics = evaluate_predictions(labels, preds, probs, model_name='ruBERT')
print('\nМетрики ruBERT на тесте:')
for k, v in metrics.items():
    if k != 'model':
        print(f'  {k:<20s}: {v:.4f}')

In [ ]:
plot_confusion_matrix(
    labels, preds,
    model_name='ruBERT',
    save_path=f'{WORKING_DIR}/cm_rubert.png',
)

In [ ]:
print(f'\nМодель сохранена: {OUTPUT_DIR}')
print(f'Предсказания сохранены: {OUTPUT_DIR}/test_probs.npy')
print(f'\nF1-macro = {metrics["f1_macro"]:.4f}')